# 02 - Limpieza de Datos y Curacion

Objetivo: documentar y validar las reglas de limpieza que convierten `ANALYTICS.OBT_TRIPS` en `ANALYTICS.OBT_TRIPS_MODEL`.

Este notebook no descarga la base completa. Las reglas estructurales ya fueron empujadas a Snowflake en `src/data/sql/01_create_obt.sql`; aqui verificamos que esas reglas quedaron aplicadas y que no se introdujo leakage.


## 1) Setup y conexion


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import snowflake.connector

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.config import get_project_config

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

cfg = get_project_config(required=True)
source_obt_table = cfg.fq_table(cfg.obt_source_table)
model_table = cfg.fq_table(cfg.obt_model_table)
train_table = cfg.fq_table(cfg.train_table)
val_table = cfg.fq_table(cfg.val_table)
test_table = cfg.fq_table(cfg.test_table)
target = cfg.target_column

print("Source OBT:", source_obt_table)
print("Model OBT:", model_table)
print("Production SQL:", PROJECT_ROOT / "src" / "data" / "sql" / "01_create_obt.sql")


Source OBT: ANALYTICS.OBT_TRIPS
Model OBT: ANALYTICS.OBT_TRIPS_MODEL
Production SQL: F:\Data mining\Proyecto Final\src\data\sql\01_create_obt.sql


In [2]:
def get_connection():
    return snowflake.connector.connect(**cfg.snowflake.connector_kwargs)


def query_df(sql: str) -> pd.DataFrame:
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            return cur.fetch_pandas_all()


## 2) Impacto de limpieza: OBT fuente vs OBT de modelado


In [3]:
row_impact_df = query_df(f"""
SELECT 'ANALYTICS.OBT_TRIPS' AS table_name,
       COUNT(*) AS rows_total,
       MIN(YEAR) AS min_year,
       MAX(YEAR) AS max_year,
       ROUND(AVG({target}), 2) AS avg_target
FROM {source_obt_table}
UNION ALL
SELECT 'ANALYTICS.OBT_TRIPS_MODEL' AS table_name,
       COUNT(*) AS rows_total,
       MIN(YEAR) AS min_year,
       MAX(YEAR) AS max_year,
       ROUND(AVG({target}), 2) AS avg_target
FROM {model_table}
ORDER BY table_name
""")
row_impact_df


,TABLE_NAME,ROWS_TOTAL,MIN_YEAR,MAX_YEAR,AVG_TARGET
0,ANALYTICS.OBT_TRIPS,844136802,2015,2025,18.6600
1,ANALYTICS.OBT_TRIPS_MODEL,844000986,2015,2025,18.6300


In [4]:
rule_impact_df = query_df(f"""
SELECT COUNT(*) AS rows_source,
       SUM(CASE WHEN {target} IS NULL THEN 1 ELSE 0 END) AS null_target,
       SUM(CASE WHEN {target} <= 0 THEN 1 ELSE 0 END) AS non_positive_target,
       SUM(CASE WHEN {target} > 500 THEN 1 ELSE 0 END) AS too_high_target,
       SUM(CASE WHEN TRIP_DISTANCE IS NULL THEN 1 ELSE 0 END) AS null_distance,
       SUM(CASE WHEN TRIP_DISTANCE <= 0 THEN 1 ELSE 0 END) AS non_positive_distance,
       SUM(CASE WHEN TRIP_DISTANCE > 100 THEN 1 ELSE 0 END) AS too_high_distance,
       SUM(CASE WHEN PASSENGER_COUNT IS NOT NULL AND (PASSENGER_COUNT < 1 OR PASSENGER_COUNT > 6) THEN 1 ELSE 0 END) AS invalid_passengers,
       SUM(CASE WHEN PU_LOCATION_ID IS NULL OR DO_LOCATION_ID IS NULL THEN 1 ELSE 0 END) AS null_locations,
       SUM(CASE WHEN PICKUP_HOUR NOT BETWEEN 0 AND 23 THEN 1 ELSE 0 END) AS invalid_pickup_hour,
       SUM(CASE WHEN DAY_OF_WEEK NOT BETWEEN 1 AND 7 THEN 1 ELSE 0 END) AS invalid_day_of_week,
       SUM(CASE WHEN SERVICE_TYPE NOT IN ('yellow', 'green') THEN 1 ELSE 0 END) AS invalid_service
FROM {source_obt_table}
""")
rule_impact_df


,ROWS_SOURCE,NULL_TARGET,NON_POSITIVE_TARGET,TOO_HIGH_TARGET,NULL_DISTANCE,NON_POSITIVE_DISTANCE,TOO_HIGH_DISTANCE,INVALID_PASSENGERS,NULL_LOCATIONS,INVALID_PICKUP_HOUR,INVALID_DAY_OF_WEEK,INVALID_SERVICE
0,844136802,0,133208,2608,0,0,0,0,0,0,0,0


## 3) Validacion de reglas finales en `OBT_TRIPS_MODEL`


In [5]:
model_quality_df = query_df(f"""
SELECT COUNT(*) AS rows_model,
       SUM(CASE WHEN {target} IS NULL OR {target} <= 0 OR {target} > 500 THEN 1 ELSE 0 END) AS invalid_target,
       SUM(CASE WHEN TRIP_DISTANCE IS NULL OR TRIP_DISTANCE <= 0 OR TRIP_DISTANCE > 100 THEN 1 ELSE 0 END) AS invalid_distance,
       SUM(CASE WHEN PASSENGER_COUNT IS NOT NULL AND (PASSENGER_COUNT < 1 OR PASSENGER_COUNT > 6) THEN 1 ELSE 0 END) AS invalid_passengers,
       SUM(CASE WHEN PU_LOCATION_ID IS NULL OR DO_LOCATION_ID IS NULL THEN 1 ELSE 0 END) AS null_locations,
       SUM(CASE WHEN PICKUP_DATETIME IS NULL OR PICKUP_DATE IS NULL THEN 1 ELSE 0 END) AS null_pickup_time,
       SUM(CASE WHEN PICKUP_HOUR NOT BETWEEN 0 AND 23 THEN 1 ELSE 0 END) AS invalid_pickup_hour,
       SUM(CASE WHEN DAY_OF_WEEK NOT BETWEEN 1 AND 7 THEN 1 ELSE 0 END) AS invalid_day_of_week,
       SUM(CASE WHEN SERVICE_TYPE NOT IN ('yellow', 'green') THEN 1 ELSE 0 END) AS invalid_service
FROM {model_table}
""")
model_quality_df


,ROWS_MODEL,INVALID_TARGET,INVALID_DISTANCE,INVALID_PASSENGERS,NULL_LOCATIONS,NULL_PICKUP_TIME,INVALID_PICKUP_HOUR,INVALID_DAY_OF_WEEK,INVALID_SERVICE
0,844000986,0,0,0,0,0,0,0,0


In [6]:
invalid_total = int(model_quality_df.drop(columns=["ROWS_MODEL"]).sum(axis=1).iloc[0])
if invalid_total != 0:
    raise ValueError(f"OBT_TRIPS_MODEL tiene reglas invalidas: {invalid_total}")
print("OK: las reglas finales de limpieza estan aplicadas en OBT_TRIPS_MODEL.")


OK: las reglas finales de limpieza estan aplicadas en OBT_TRIPS_MODEL.


## 4) Nulos y categorias de baja calidad


In [7]:
null_profile_df = query_df(f"""
SELECT COUNT(*) AS rows_model,
       SUM(CASE WHEN VENDOR_ID IS NULL THEN 1 ELSE 0 END) AS null_vendor_id,
       SUM(CASE WHEN RATE_CODE_ID IS NULL THEN 1 ELSE 0 END) AS null_rate_code_id,
       SUM(CASE WHEN TRIP_TYPE IS NULL THEN 1 ELSE 0 END) AS null_trip_type,
       SUM(CASE WHEN PASSENGER_COUNT IS NULL THEN 1 ELSE 0 END) AS null_passenger_count,
       SUM(CASE WHEN PU_BOROUGH IS NULL THEN 1 ELSE 0 END) AS null_pu_borough,
       SUM(CASE WHEN DO_BOROUGH IS NULL THEN 1 ELSE 0 END) AS null_do_borough,
       SUM(CASE WHEN PU_ZONE IS NULL THEN 1 ELSE 0 END) AS null_pu_zone,
       SUM(CASE WHEN DO_ZONE IS NULL THEN 1 ELSE 0 END) AS null_do_zone,
       SUM(CASE WHEN VENDOR_NAME = 'Unknown' THEN 1 ELSE 0 END) AS unknown_vendor,
       SUM(CASE WHEN RATE_CODE_DESC = 'Unknown' THEN 1 ELSE 0 END) AS unknown_rate_code,
       SUM(CASE WHEN TRIP_TYPE_DESC = 'Unknown' THEN 1 ELSE 0 END) AS unknown_trip_type
FROM {model_table}
""")
null_profile_df


,ROWS_MODEL,NULL_VENDOR_ID,NULL_RATE_CODE_ID,NULL_TRIP_TYPE,NULL_PASSENGER_COUNT,NULL_PU_BOROUGH,NULL_DO_BOROUGH,NULL_PU_ZONE,NULL_DO_ZONE,UNKNOWN_VENDOR,UNKNOWN_RATE_CODE,UNKNOWN_TRIP_TYPE
0,844000986,0,19046545,779984972,19046545,0,0,0,0,924221,20567388,779984972


In [8]:
cardinality_df = query_df(f"""
SELECT 'SERVICE_TYPE' AS column_name, COUNT(DISTINCT SERVICE_TYPE) AS distinct_values FROM {model_table}
UNION ALL SELECT 'VENDOR_ID', COUNT(DISTINCT VENDOR_ID) FROM {model_table}
UNION ALL SELECT 'RATE_CODE_ID', COUNT(DISTINCT RATE_CODE_ID) FROM {model_table}
UNION ALL SELECT 'TRIP_TYPE', COUNT(DISTINCT TRIP_TYPE) FROM {model_table}
UNION ALL SELECT 'PU_LOCATION_ID', COUNT(DISTINCT PU_LOCATION_ID) FROM {model_table}
UNION ALL SELECT 'DO_LOCATION_ID', COUNT(DISTINCT DO_LOCATION_ID) FROM {model_table}
UNION ALL SELECT 'LOCATION_PAIR', COUNT(DISTINCT LOCATION_PAIR) FROM {model_table}
ORDER BY distinct_values DESC
""")
cardinality_df


,COLUMN_NAME,DISTINCT_VALUES
0,LOCATION_PAIR,63846
1,PU_LOCATION_ID,265
2,DO_LOCATION_ID,265
3,RATE_CODE_ID,7
4,VENDOR_ID,6
5,SERVICE_TYPE,2
6,TRIP_TYPE,2


In [9]:
top_categories_df = query_df(f"""
SELECT 'PU_BOROUGH' AS column_name, COALESCE(PU_BOROUGH, 'NULL') AS value, COUNT(*) AS rows_total
FROM {model_table}
GROUP BY 1, 2
UNION ALL
SELECT 'DO_BOROUGH' AS column_name, COALESCE(DO_BOROUGH, 'NULL') AS value, COUNT(*) AS rows_total
FROM {model_table}
GROUP BY 1, 2
UNION ALL
SELECT 'RATE_CODE_DESC' AS column_name, COALESCE(RATE_CODE_DESC, 'NULL') AS value, COUNT(*) AS rows_total
FROM {model_table}
GROUP BY 1, 2
ORDER BY column_name, rows_total DESC
""")
top_categories_df.head(40)


,COLUMN_NAME,VALUE,ROWS_TOTAL
0,DO_BOROUGH,Manhattan,710783577
1,DO_BOROUGH,Queens,56876055
2,DO_BOROUGH,Brooklyn,55115728
3,DO_BOROUGH,Bronx,9801447
4,DO_BOROUGH,Unknown,7627733
5,DO_BOROUGH,N/A,2055544
6,DO_BOROUGH,EWR,1520765
7,DO_BOROUGH,Staten Island,220137
8,PU_BOROUGH,Manhattan,725340711
9,PU_BOROUGH,Queens,70119888


## 5) Validacion contra data leakage


In [10]:
leakage_columns = [
    'DROPOFF_DATETIME', 'DROPOFF_DATE', 'DROPOFF_HOUR',
    'FARE_AMOUNT', 'EXTRA', 'MTA_TAX', 'TIP_AMOUNT', 'TOLLS_AMOUNT',
    'IMPROVEMENT_SURCHARGE', 'CONGESTION_SURCHARGE', 'AIRPORT_FEE',
    'TRIP_DURATION_MIN', 'AVG_SPEED_MPH', 'TIP_PCT',
    'PAYMENT_TYPE', 'PAYMENT_TYPE_DESC', 'RUN_ID', 'INGESTED_AT_UTC',
    'SOURCE_SERVICE', 'SOURCE_YEAR', 'SOURCE_MONTH'
]
leakage_sql = ", ".join(f"'{col}'" for col in leakage_columns)

leakage_check_df = query_df(f"""
SELECT COLUMN_NAME
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = '{cfg.analytics_schema.upper()}'
  AND TABLE_NAME = '{cfg.obt_model_table.upper()}'
  AND COLUMN_NAME IN ({leakage_sql})
ORDER BY COLUMN_NAME
""")

if leakage_check_df.empty:
    print("OK: no hay columnas de leakage en OBT_TRIPS_MODEL.")
else:
    display(leakage_check_df)
    raise ValueError("Existen columnas con leakage en OBT_TRIPS_MODEL.")


OK: no hay columnas de leakage en OBT_TRIPS_MODEL.


## 6) Reglas pandas equivalentes para muestras

La limpieza productiva se ejecuta en Snowflake. Esta funcion pandas solo sirve para validar o depurar muestras en notebooks.


In [11]:
SAFE_FEATURE_COLUMNS = [
    'SERVICE_TYPE', 'VENDOR_ID', 'RATE_CODE_ID', 'TRIP_TYPE',
    'PASSENGER_COUNT', 'TRIP_DISTANCE', 'PICKUP_HOUR', 'DAY_OF_WEEK',
    'MONTH', 'YEAR', 'IS_WEEKEND', 'PICKUP_TIME_BAND',
    'PU_LOCATION_ID', 'DO_LOCATION_ID', 'PU_BOROUGH', 'DO_BOROUGH',
    'SAME_BOROUGH_FLAG', 'AIRPORT_TRIP_FLAG', 'LOCATION_PAIR'
]
TRACEABILITY_COLUMNS = ['TRIP_NK', 'PICKUP_DATETIME', 'PICKUP_DATE']
TARGET_COLUMN = target


def clean_model_sample(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned = cleaned[cleaned[TARGET_COLUMN].notna()]
    cleaned = cleaned[(cleaned[TARGET_COLUMN] > 0) & (cleaned[TARGET_COLUMN] <= 500)]
    cleaned = cleaned[cleaned['TRIP_DISTANCE'].notna()]
    cleaned = cleaned[(cleaned['TRIP_DISTANCE'] > 0) & (cleaned['TRIP_DISTANCE'] <= 100)]
    cleaned = cleaned[cleaned['PU_LOCATION_ID'].notna() & cleaned['DO_LOCATION_ID'].notna()]
    cleaned = cleaned[cleaned['SERVICE_TYPE'].isin(['yellow', 'green'])]
    cleaned = cleaned[(cleaned['PASSENGER_COUNT'].isna()) | (cleaned['PASSENGER_COUNT'].between(1, 6))]
    cleaned = cleaned[cleaned['PICKUP_HOUR'].between(0, 23)]
    cleaned = cleaned[cleaned['DAY_OF_WEEK'].between(1, 7)]
    return cleaned

sample_validation_df = query_df(f"""
SELECT *
FROM {model_table} SAMPLE BERNOULLI (0.001)
LIMIT 10000
""")

sample_after_cleaning = clean_model_sample(sample_validation_df)
print("sample before:", sample_validation_df.shape)
print("sample after:", sample_after_cleaning.shape)

assert len(sample_after_cleaning) == len(sample_validation_df), "La muestra modelable no deberia perder filas con las reglas finales."
print("OK: reglas pandas equivalentes preservan la muestra ya limpia.")


sample before: (8548, 28)
sample after: (8548, 28)
OK: reglas pandas equivalentes preservan la muestra ya limpia.


## 7) Confirmacion de splits temporales


In [12]:
split_df = query_df(f"""
SELECT 'TRAIN_SET' AS split_name, COUNT(*) AS rows_split, MIN(YEAR) AS min_year, MAX(YEAR) AS max_year
FROM {train_table}
UNION ALL
SELECT 'VAL_SET' AS split_name, COUNT(*) AS rows_split, MIN(YEAR) AS min_year, MAX(YEAR) AS max_year
FROM {val_table}
UNION ALL
SELECT 'TEST_SET' AS split_name, COUNT(*) AS rows_split, MIN(YEAR) AS min_year, MAX(YEAR) AS max_year
FROM {test_table}
ORDER BY split_name
""")
split_df


,SPLIT_NAME,ROWS_SPLIT,MIN_YEAR,MAX_YEAR
0,TEST_SET,44109224,2025,2025
1,TRAIN_SET,760168027,2015,2023
2,VAL_SET,39723735,2024,2024


## 8) Decisiones para el notebook 03

Decisiones tomadas:
- Usar `ANALYTICS.OBT_TRIPS_MODEL` como fuente unica para modelado.
- Mantener splits temporales en Snowflake: train 2015-2023, validacion 2024, test 2025.
- No exportar datasets intermedios pesados a `data/interim`.
- No usar columnas post-viaje ni componentes directos del precio.
- En feature engineering, transformar categoricas con `OneHotEncoder` o codificacion controlada para columnas de baja cardinalidad.
- Para zonas y pares de zonas, evaluar top-N/Other o codificacion por frecuencia para evitar matrices demasiado grandes.
- El `TEST_SET` permanece reservado para el resultado final del modelo ganador.
